# OhamLab VoiceStudio · Kaggle Hindi Trainer
**GPU training pilot** — download the base model, load your example Hindi dataset, select a speaker, prepare, train, export and listen. The training code lives inside **your OhamLab repository**; no external fine-tuning Git install.

**Before running:** Kaggle → Settings → Accelerator **GPU**, Internet **On**. This notebook is an experimental smoke test, not a verified training release. A tiny dataset will not produce a high-quality voice.

## 1 · Easy settings — edit only this cell

In [ ]:
BASE_MODEL = "rahul7star/tts-nano-o-hi"
DATASET_ID = "rahul7star/hindi_dataset_small_example"
REPO_URL = "https://github.com/innokria/ohamlab-voicestudio.git"
# Set to a numeric speaker_id to pick one speaker, or None to auto-select the most frequent.
SPEAKER_ID = None
VOICE_NAME = "OhamHindi"
TRAIN_STEPS = 100
BATCH_FRAMES = 1200  # conservative T4 pilot
SAVE_EVERY = 50
VALID_EVERY = 50
RESUME = False
RUN_EXPORT = True
UPLOAD_TO_HF = False
HF_OUTPUT_REPO = ""  # example: yourname/ohamlab-hindi-test
TEST_SENTENCE = "नमस्ते! यह मेरी नई हिंदी आवाज़ का परीक्षण है।"
WORK = "/kaggle/working/ohamlab"

## 2 · GPU and OhamLab installation

In [ ]:
import os, sys, subprocess, pathlib
import torch
print("Python:", sys.version.split()[0], "Torch:", torch.__version__)
assert torch.cuda.is_available(), "Enable GPU in Kaggle Settings before training."
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+" + REPO_URL], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets>=2.19,<4", "soundfile", "scipy", "librosa", "sentencepiece", "safetensors", "pyyaml", "speechbrain>=1.0", "torchaudio"], check=True)

**Important:** Before you run this notebook, push the merged training modules to your GitHub repository. The existing published repository's older version does not include them. You can also upload the merged ZIP as a Kaggle Dataset and install locally instead.

## 3 · Download model and inspect dataset

In [ ]:
from huggingface_hub import snapshot_download
from datasets import load_dataset, Audio
from collections import Counter
base = snapshot_download(repo_id=BASE_MODEL, local_dir=f"{WORK}/base")
print("Model files:", sorted(p.name for p in pathlib.Path(base).iterdir()))
assert (pathlib.Path(base)/"hima_tts").exists(), "Base model has no hima_tts source; training backend needs compatible inference code."
assert (pathlib.Path(base)/"config.yaml").exists(), "Missing config.yaml in base model."
dset = load_dataset(DATASET_ID, split="train")
print("Rows:", len(dset), "Columns:", dset.column_names)
for c in ("audio", "transcription", "speaker_id"):
    assert c in dset.column_names, f"Missing expected column {c}"
print("Speaker counts:", Counter(str(x) for x in dset["speaker_id"]))

## 4 · Select speaker, preview recordings and prepare audio

In [ ]:
import numpy as np, soundfile as sf
from scipy.signal import resample_poly
from math import gcd
from IPython.display import Audio as AudioPlayer, display
from collections import Counter
counts = Counter(str(x) for x in dset["speaker_id"])
chosen = str(SPEAKER_ID) if SPEAKER_ID is not None else counts.most_common(1)[0][0]
subset = dset.filter(lambda row: str(row["speaker_id"]) == chosen)
print(f"Selected speaker {chosen}: {len(subset)} clips")
assert len(subset) >= 3, "Need at least 3 clips for a useful train/validation pilot. Choose another speaker or larger dataset."
# Force dataset decoding to numpy arrays; this may require audio decoding dependencies on Kaggle.
subset = subset.cast_column("audio", Audio(decode=True))
out = pathlib.Path(WORK)/"clips";out.mkdir(parents=True, exist_ok=True)
rows=[]
for i, item in enumerate(subset):
    text=str(item["transcription"]).strip()
    if not text: continue
    sample=item["audio"]
    audio=np.asarray(sample["array"],dtype=np.float32)
    if audio.ndim==2: audio=audio.mean(axis=0 if audio.shape[0] < audio.shape[1] else 1)
    sr=int(sample["sampling_rate"])
    if sr!=24000:
        g=gcd(sr,24000);audio=resample_poly(audio,24000//g,sr//g).astype(np.float32)
    duration=len(audio)/24000
    if not (0.7<=duration<=29):continue
    name=f"clip_{i:04d}.wav";sf.write(out/name,audio,24000)
    rows.append((name,text,VOICE_NAME))
assert len(rows)>=3, f"Only {len(rows)} valid clips after filtering; need at least 3."
with (out/"metadata.csv").open("w",encoding="utf-8") as f:
    for path,text,voice in rows:
        f.write(f"{path}|{text.replace('|',' ')}|{voice}\n")
print("Prepared",len(rows),"clips. Sample:",rows[0][1]);display(AudioPlayer(filename=str(out/rows[0][0])))

## 5 · Encode dataset for training
This uses the codec and voice embedding model supplied by the compatible base-model runtime. It can take several minutes on first run.

In [ ]:
cmd=[sys.executable,"-m","ohamlab_voicestudio.training.backend.prepare","--base",str(base),"--data",str(out),"--work",WORK]
print("Running:"," ".join(cmd));subprocess.run(cmd,check=True)

## 6 · Fine-tune LoRA adapter (Kaggle GPU)

In [ ]:
cmd=[sys.executable,"-m","ohamlab_voicestudio.training.backend.train","--base",str(base),"--work",WORK,
     "--steps",str(TRAIN_STEPS),"--batch_frames",str(BATCH_FRAMES),"--save_every",str(SAVE_EVERY),"--valid_every",str(VALID_EVERY)]
if RESUME: cmd.append("--resume")
print("Starting training; progress is saved under",WORK+"/run")
subprocess.run(cmd,check=True)

## 7 · Export a compatible voice model and listen

In [ ]:
export_dir=pathlib.Path(WORK)/"exported-model"
if RUN_EXPORT:
    cmd=[sys.executable,"-m","ohamlab_voicestudio.training.backend.export","--base",str(base),"--work",WORK,"--out",str(export_dir),"--samples",TEST_SENTENCE]
    subprocess.run(cmd,check=True)
    wav=export_dir/"samples"/f"{VOICE_NAME}.wav"
    assert wav.exists(), f"No sample was exported: {wav}"
    display(AudioPlayer(filename=str(wav)))
else:
    print("Export skipped")

## 8 · Download results / optional private Hugging Face upload

In [ ]:
import shutil
if export_dir.exists():
    archive=shutil.make_archive("/kaggle/working/ohamlab-hindi-model","zip",root_dir=export_dir)
    print("Download from Kaggle Output:",archive)
if UPLOAD_TO_HF:
    assert HF_OUTPUT_REPO, "Set HF_OUTPUT_REPO in settings"
    from huggingface_hub import HfApi
    token=os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as e:
            raise RuntimeError("Add HF_TOKEN as a Kaggle secret; never paste it into the notebook") from e
    api=HfApi(token=token)
    api.create_repo(repo_id=HF_OUTPUT_REPO,repo_type="model",private=True,exist_ok=True)
    api.upload_folder(folder_path=str(export_dir),repo_id=HF_OUTPUT_REPO,repo_type="model")
    print("Uploaded privately:",HF_OUTPUT_REPO)

### Troubleshooting
- **No `hima_tts` in model folder:** The selected model repository does not contain the inference source expected by the training backend. Stop and inspect the base model; do not substitute unrelated weights.
- **Too few clips:** This example dataset is tiny. Choose a speaker with more recordings or a larger dataset for meaningful results.
- **Out of memory:** Reduce `BATCH_FRAMES` to 800, then restart from the prepare stage.
- **Notebook restarts:** Kaggle `/kaggle/working` may not survive new sessions; save the work directory to a Kaggle Dataset if you need resume across sessions.
- **Licensing:** Training modules include MIT-licensed code with preserved notices under `LICENSES/`. Model and voice redistribution rights must be checked separately.